# QIBVT — Google Colab Analysis
Run from top to bottom. Public data is downloaded live; verify company figures against official filings.

In [ ]:
!pip -q install yfinance pandas numpy matplotlib plotly

In [ ]:
import pandas as pd, numpy as np, yfinance as yf, matplotlib.pyplot as plt
from pathlib import Path
TICKERS = ['AAPL','MSFT','NVDA','TCS.NS','RELIANCE.NS']
BENCHMARKS = {'US':'SPY','India':'^NSEI'}
prices = yf.download(TICKERS, period='5y', auto_adjust=True, progress=False)['Close']
prices = prices.dropna(how='all').ffill()
prices.tail()

## 1. Market performance
CAGR, volatility and max drawdown are descriptive historical statistics, not forecasts.

In [ ]:
returns = prices.pct_change().dropna(how='all')
rows=[]
for t in prices.columns:
    p=prices[t].dropna(); r=returns[t].dropna()
    if len(p)>1 and len(r)>1:
        years=len(r)/252
        rows.append({'Ticker':t,'Last Price':p.iloc[-1],
                     'Total Return':p.iloc[-1]/p.iloc[0]-1,
                     'CAGR':(p.iloc[-1]/p.iloc[0])**(1/years)-1,
                     'Annualised Volatility':r.std()*np.sqrt(252),
                     'Max Drawdown':(p/p.cummax()-1).min()})
market_metrics=pd.DataFrame(rows).set_index('Ticker')
display(market_metrics.style.format('{:.2%}'))

In [ ]:
(prices / prices.iloc[0] * 100).plot(figsize=(13,6), title='Growth of 100')
plt.ylabel('Indexed adjusted price'); plt.grid(True); plt.show()

## 2. Annual financial statements
Yahoo Finance fields can be missing or inconsistently labelled. The code below exports what the provider returns; cross-check each value against annual reports / SEC filings.

In [ ]:
def annual_statement_snapshot(ticker, years=3):
    y=yf.Ticker(ticker)
    inc, bal, cf = y.income_stmt, y.balance_sheet, y.cashflow
    aliases={'Revenue':(inc,['Total Revenue','Operating Revenue']),
             'EBIT':(inc,['EBIT','Operating Income']),
             'Net Income':(inc,['Net Income','Net Income Common Stockholders']),
             'Total Debt':(bal,['Total Debt','Long Term Debt And Capital Lease Obligation']),
             'Cash':(bal,['Cash And Cash Equivalents','Cash Cash Equivalents And Short Term Investments']),
             'Operating Cash Flow':(cf,['Operating Cash Flow','Cash Flow From Continuing Operating Activities']),
             'CapEx':(cf,['Capital Expenditure','Capital Expenditures']),
             'Diluted Shares':(inc,['Diluted Average Shares','Diluted Average Shares Outstanding'])}
    cols=list(inc.columns[:years]) if inc is not None and not inc.empty else []
    out=[]
    for c in cols:
        row={'Ticker':ticker,'Fiscal Year End':str(pd.Timestamp(c).date())}
        for field,(df,names) in aliases.items():
            value=np.nan
            if df is not None and not df.empty and c in df.columns:
                for name in names:
                    if name in df.index and pd.notna(df.loc[name,c]):
                        value=float(df.loc[name,c]); break
            row[field]=abs(value) if field=='CapEx' and pd.notna(value) else value
        out.append(row)
    result=pd.DataFrame(out)
    if not result.empty:
        result['EBIT Margin']=result['EBIT']/result['Revenue']
        result['Free Cash Flow']=result['Operating Cash Flow']-result['CapEx']
        result['Net Debt']=result['Total Debt']-result['Cash']
    return result

financials=pd.concat([annual_statement_snapshot(t) for t in TICKERS], ignore_index=True)
display(financials)
financials.to_csv('qibvt_financial_statements.csv', index=False)

## 3. Trading comparables
Multiples may be unavailable or stale. Peer selection must be justified. Reliance is diversified and should ideally be valued using segment-level sum-of-the-parts.

In [ ]:
PEERS={'AAPL':['MSFT','DELL','HPQ'],'MSFT':['ORCL','GOOGL','AMZN'],
       'NVDA':['AMD','AVGO','QCOM'],'TCS.NS':['INFY.NS','WIPRO.NS','HCLTECH.NS'],
       'RELIANCE.NS':['ADANIENT.NS','ITC.NS','BHARTIARTL.NS']}
all_companies=sorted(set(TICKERS+[x for v in PEERS.values() for x in v]))
comp_rows=[]
for t in all_companies:
    try:
        i=yf.Ticker(t).info or {}
        comp_rows.append({'Ticker':t,'Company':i.get('shortName') or i.get('longName'),
          'Currency':i.get('currency'),'Market Cap':i.get('marketCap'),
          'Enterprise Value':i.get('enterpriseValue'),'P/E':i.get('trailingPE'),
          'Forward P/E':i.get('forwardPE'),'P/B':i.get('priceToBook'),
          'EV/EBITDA':i.get('enterpriseToEbitda'),'EV/Revenue':i.get('enterpriseToRevenue')})
    except Exception as e: comp_rows.append({'Ticker':t,'Error':str(e)})
comps=pd.DataFrame(comp_rows)
display(comps)
comps.to_csv('qibvt_trading_comps.csv',index=False)

## 4. Beta and CAPM
Use market-matched benchmarks and currency-consistent risk-free rates. The example uses a 4% US and 7% India risk-free assumption; replace with dated government-bond yields.

In [ ]:
bench_prices=yf.download(['SPY','^NSEI'],period='5y',auto_adjust=True,progress=False)['Close']
bench_returns=bench_prices.pct_change()
def beta_capm(ticker, benchmark, rf):
    a=prices[ticker].pct_change().rename('asset')
    m=bench_returns[benchmark].rename('market')
    z=pd.concat([a,m],axis=1).dropna()
    beta=z.asset.cov(z.market)/z.market.var()
    rm=(1+z.market.mean())**252-1
    return {'Ticker':ticker,'Benchmark':benchmark,'Beta':beta,'Risk-free':rf,
            'Annualised market return':rm,'CAPM cost of equity':rf+beta*(rm-rf)}
capm_rows=[]
for t in TICKERS:
    b='^NSEI' if t.endswith('.NS') else 'SPY'
    capm_rows.append(beta_capm(t,b,.07 if b=='^NSEI' else .04))
display(pd.DataFrame(capm_rows))

## 5. Monte Carlo and historical VaR
GBM is a simplified scenario model. VaR is not the maximum possible loss.

In [ ]:
def gbm_terminal(ticker, n=10000, days=252, seed=42):
    p=prices[ticker].dropna(); r=p.pct_change().dropna()
    mu=r.mean()*252; vol=r.std()*np.sqrt(252); s0=p.iloc[-1]
    rng=np.random.default_rng(seed); z=rng.standard_normal(n)
    terminal=s0*np.exp((mu-.5*vol**2)*days/252+vol*np.sqrt(days/252)*z)
    return terminal,mu,vol
def hist_var(ticker, notional=1_000_000, confidence=.95):
    return -returns[ticker].dropna().quantile(1-confidence)*notional
chosen='AAPL'
terminal,mu,vol=gbm_terminal(chosen)
print('Annualised drift:',mu,'Annualised volatility:',vol)
print('Terminal percentiles:',np.percentile(terminal,[5,50,95]))
print('1-day 95% historical VaR on 1m notional:',hist_var(chosen))
plt.figure(figsize=(10,5)); plt.hist(terminal,bins=60); plt.title(f'{chosen}: 10,000 one-year terminal price scenarios'); plt.show()

## 6. Export outputs

In [ ]:
market_metrics.to_csv('qibvt_market_metrics.csv')
print('CSV files created. Download them from the Colab Files panel.')